# Step 3c: LightGBM ranker (supplementary)

> **Supplementary, not part of the report.** This ranker was explored before the main model became a deep network and is kept for reference. The report's comparisons and the final evaluation (step 6) do not use it.

A non-deep comparison for the main deep model (step 4): a gradient-boosted tree ranker that scores every (user, candidate) pair from four groups of engineered features (feature schema contract v0.1), then ranks candidates within each impression.

| Group | Inputs | Role |
|---|---|---|
| T1, article behaviour | prior exposures and clicks, prior CTR, evidence weight, 6-hour exposures/clicks, CTR momentum, age | What the candidate's own history says |
| T2, content-transferred behaviour | recent CTR and momentum of the same category, subcategory and entities; semantic-neighbour CTR | Behaviour borrowed from related articles, available even when the candidate has none |
| T3, pure content | category and subcategory codes, entity count and coverage, 32 title PCs, 16 entity PCs | What the article is about |
| U, user–article matching | history length, category/subcategory affinity, text and entity similarity, entity overlap, user profile PCs | How well the candidate fits this user; kept in every ablation |

```
meta + features (row tables) ─┐
article_vectors ── PCA (fit on training-accessible articles) ─┼─ join by news_id ──┐
user_vectors ───── same PCA ──────────────────────────────────┴─ join by impression ─┼─► LightGBM ─► score ─► rank within impression
                                                                                     │
                                                  early stopping on validation ◄────┘
```

**Objective.** LambdaRank, which optimises nDCG directly within each impression (rows grouped by impression), with early stopping on validation nDCG@10. Its scores rank candidates within an impression; they are not click probabilities. A binary log-loss model with the same columns and tree shape is trained afterwards as a comparison. (The interim plan made the binary model the main model. LambdaRank scored higher on every validation metric in the first run on real features, so it was promoted; this is a deliberate change.)

**No negative sampling.** All candidates are kept for training. At 1 click per 24 non-clicks this fits comfortably in free-Colab memory, and the trees see the real class balance. (The interim report anticipated 1:4 sampling; this is a deliberate change.)

**Tuning.** Four tree-shape settings are trained on the train split and compared on validation only. Selection rule: highest validation Cold Recall@10 among settings within 0.005 validation nDCG@10 of the best. The tolerance is provisional until the team agrees it. The test split is scored once, after selection.

**Memory.** Embedding vectors stay in their own tables. Each article and user is reduced once with PCA, cached, and joined onto the rows as float32 columns.

> Metrics here come from `shared/eval_interim.py`, a stand-in used for tuning until the team evaluator exists. The numbers in the final report come from `step6_evaluation_diagnostics.ipynb`.

**At a glance**

| | |
|---|---|
| Reads | `features/v0.1/` on Google Drive `MyDrive/BT4222_G11/`; MIND-small histories via `kagglehub` |
| Writes | `predictions/real/` (`lgbm_full`, `lgbm_full_binary`); `models/real/lgbm_selected_params.json` |
| Time and hardware | About 10 minutes on CPU |
| Run after | Step 2 (steps 3a and 3b for the comparison table) |

## 1. Setup

In [1]:
import os, sys, subprocess, json, time

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Bryng23/BT4222-Group-11.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/BT4222-Group-11"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    DATA_ROOT = "/content/drive/MyDrive/BT4222_G11"
else:
    REPO_DIR = os.environ.get("G11_REPO_DIR", os.getcwd())
    DATA_ROOT = os.environ.get("G11_DATA_ROOT", os.path.join(REPO_DIR, "data"))
sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
from shared import schema as S, feature_groups as FG, lgbm_pipeline as L, selection as SEL
from shared.predictions import write_predictions
from shared import eval_interim as E      # swap for the team evaluator once it exists

DATA_KIND = "REAL"   # recorded in every prediction file
import kagglehub
RAW_DIR = os.path.join(kagglehub.dataset_download("arashnic/mind-news-dataset"), "MINDsmall_train")
FEATURE_ROOT = os.path.join(DATA_ROOT, "features", f"v{S.SCHEMA_VERSION}")
version = S.read_version(FEATURE_ROOT)
assert version.get("data_kind", "REAL") == DATA_KIND, f"feature tables are {version.get('data_kind')}, run is {DATA_KIND}"
PRED_ROOT = os.path.join(DATA_ROOT, "predictions")
MODEL_DIR = os.path.join(DATA_ROOT, "models", DATA_KIND.lower())
os.makedirs(MODEL_DIR, exist_ok=True)
SEED = 4222
N_TITLE_PC, N_ENT_PC = 32, 16
INCLUDE_USER_PC = True      # user profile vectors projected with the article PCA; set False if RAM runs short
SHOW = ["auc", "mrr", "ndcg@5", "ndcg@10", "cold_recall@10", "cold_hit@10", "warm_recall@10"]
print(f"DATA_KIND={DATA_KIND} | features: {FEATURE_ROOT} | schema {version['schema_version']}")

C:\Users\Bryan Ng\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DATA_KIND=REAL | features: C:\Users\Bryan Ng\g11_data\features\v0.1 | schema 0.1


## 2. Reduce vectors and load the row tables

PCA sees only articles available during training: training candidates and every article in the click histories of training impressions. Validation and test articles are projected with the frozen components.

In [2]:
t0 = time.time()
train_meta_ids = pd.read_parquet(S.meta_path(FEATURE_ROOT, "train"), columns=["impression_id", "news_id"])
accessible = set(train_meta_ids["news_id"].astype(str)) | L.train_histories(RAW_DIR, train_meta_ids["impression_id"].unique())
reducer = L.VectorReducer(N_TITLE_PC, N_ENT_PC, seed=SEED).fit(FEATURE_ROOT, accessible)
print(f"PCA fitted on {reducer.n_fit_articles:,} training-accessible articles "
      f"({reducer.n_fit_entity_articles:,} with entity vectors); explained variance {reducer.explained_variance()}")

data = {split: L.load_split(FEATURE_ROOT, split, reducer, include_user_pc=INCLUDE_USER_PC) for split in S.SPLITS}
INCLUDE_SEMANTIC = "semantic_neighbor_ctr" in data["train"][1].columns   # optional in the contract
GROUPS = FG.group_columns(N_TITLE_PC, N_ENT_PC, include_user_pc=INCLUDE_USER_PC, include_semantic=INCLUDE_SEMANTIC)
print("semantic_neighbor_ctr present:", INCLUDE_SEMANTIC)
pd.DataFrame({split: {"rows": len(m), "impressions": m["impression_id"].nunique(), "columns available": x.shape[1],
                      "memory MB": round(x.memory_usage(deep=True).sum() / 1e6, 1)}
              for split, (m, x) in data.items()}).T

PCA fitted on 43,370 training-accessible articles (37,368 with entity vectors); explained variance {'title': 0.4252181649208069, 'entity': 0.7117447853088379}


semantic_neighbor_ctr present: True


,rows,impressions,columns available,memory MB
train,3389342.0,95071.0,131.0,1776.0
val,1231673.0,31624.0,131.0,645.4
test,1222429.0,30270.0,131.0,640.6


## 3. Tune tree shape on validation

In [3]:
FULL_COLS = FG.feature_columns("full", n_title_pc=N_TITLE_PC, n_ent_pc=N_ENT_PC, include_user_pc=INCLUDE_USER_PC,
                         include_semantic=INCLUDE_SEMANTIC)
(m_tr, x_tr), (m_va, x_va) = data["train"], data["val"]
X_tr, X_va = L.design(x_tr, FULL_COLS), L.design(x_va, FULL_COLS)
g_tr, g_va = L.impression_groups(m_tr), L.impression_groups(m_va)   # LambdaRank ranks within impressions
print(f"full model: {len(FULL_COLS)} columns")

boosters, rows = {}, []
for params in L.PARAM_GRID:
    pid = L.param_id(params)
    t0 = time.time()
    b = L.fit(X_tr, m_tr["label"], X_va, m_va["label"], params, seed=SEED, group_tr=g_tr, group_va=g_va)
    boosters[pid] = b
    res = E.evaluate(m_va, L.predict(b, X_va))
    rows.append({"params": pid, "best_iteration": b.best_iteration, "minutes": round((time.time() - t0) / 60, 1),
                 **{k: res[k] for k in SHOW}})
grid = pd.DataFrame(rows).set_index("params")
SELECTED = SEL.select(grid)
selected_params = next(p for p in L.PARAM_GRID if L.param_id(p) == SELECTED)
print(f"selected {SELECTED}: {SEL.describe()}")
with open(os.path.join(MODEL_DIR, "lgbm_selected_params.json"), "w") as f:
    json.dump({"objective": L.BASE_PARAMS["objective"], "params": selected_params, "best_iteration": boosters[SELECTED].best_iteration,
               "n_title_pc": N_TITLE_PC, "n_ent_pc": N_ENT_PC, "include_user_pc": INCLUDE_USER_PC}, f, indent=1)
grid.round(4)

full model: 122 columns


selected num_leaves=31,min_data_in_leaf=500: highest validation cold_recall@10 among settings whose validation ndcg@10 is within 0.005 of the best; ties broken on ndcg@10


,best_iteration,minutes,auc,mrr,ndcg@5,ndcg@10,cold_recall@10,cold_hit@10,warm_recall@10
params,,,,,,,,,
"num_leaves=31,min_data_in_leaf=100",401,1.3,0.7174,0.3630,0.3965,0.4533,0.6497,0.6631,0.6868
"num_leaves=31,min_data_in_leaf=500",432,1.4,0.7166,0.3630,0.3968,0.4533,0.6507,0.6631,0.6870
"num_leaves=127,min_data_in_leaf=100",197,1.1,0.7153,0.3623,0.3957,0.4523,0.6284,0.6439,0.6860
"num_leaves=127,min_data_in_leaf=500",136,1.0,0.7156,0.3604,0.3938,0.4506,0.6356,0.6493,0.6851


**Takeaway: tuning.** The four tree shapes are within 0.003 validation nDCG@10 of each other, so tree shape barely matters for overall ranking. Cold Recall@10 varies more (0.63 to 0.65). The rule picks `num_leaves=31, min_data_in_leaf=500`, which ties for the best nDCG@10 (0.453) and has the highest Cold Recall@10 (0.651).

## 4. Score every split with the selected model

Train, validation and test are scored on full slates and written as prediction files. This is the only time the test split is scored.

In [4]:
full = boosters[SELECTED]
final, main_scores = {}, {}
for split, (m, x) in data.items():
    sc = main_scores[split] = L.predict(full, L.design(x, FULL_COLS))
    write_predictions(m["row_id"], sc, root=PRED_ROOT, data_kind=DATA_KIND, model_id="lgbm_full", config_id=SELECTED,
                      split=split, schema_version=S.SCHEMA_VERSION,
                      extra={"objective": "lambdarank", "best_iteration": full.best_iteration, "selected_on": "val"})
    final[split] = E.evaluate(m, sc)
pd.DataFrame(final).T[SHOW + ["n_cold_click_impressions"]].round(4)

,auc,mrr,ndcg@5,ndcg@10,cold_recall@10,cold_hit@10,warm_recall@10,n_cold_click_impressions
train,0.7768,0.4503,0.4957,0.5456,0.6943,0.7111,0.7636,3223.0
val,0.7166,0.3630,0.3968,0.4533,0.6507,0.6631,0.6870,938.0
test,0.7141,0.3376,0.3695,0.4306,0.6054,0.6237,0.6727,837.0


**Takeaway: splits.** Train is clearly above validation and test (nDCG@10 0.546 vs 0.453 and 0.431). Part of that is fit to the training days and part is easier train slates (random scores 0.307 on train, 0.262 on test). From validation to test, nDCG@10 drops a little (0.453 → 0.431) and Cold Recall@10 drops more (0.651 → 0.605).

## 5. Comparison with the step 3a and 3b baselines

Reads whichever baseline prediction files exist for this data kind and evaluates them on the same rows with the same metric code.

In [5]:
import glob
from shared.predictions import read_predictions
cmp_rows = []
for split in ("val", "test"):
    m = data[split][0]
    cmp_rows.append({"split": split, "model": "lgbm_full", **{k: final[split][k] for k in SHOW}})
    for path in sorted(glob.glob(os.path.join(PRED_ROOT, DATA_KIND.lower(), f"{DATA_KIND}__*__{split}.parquet"))):
        p, info = read_predictions(path)
        if info.get("selected_on") != "val" or info["model_id"].startswith("lgbm"):
            continue   # only the frozen baseline choices
        sc = m[["row_id"]].merge(p, on="row_id", how="left")["predicted_score"]
        if sc.isna().any():
            continue
        cmp_rows.append({"split": split, "model": info["model_id"], **{k: E.evaluate(m, sc.to_numpy())[k] for k in SHOW}})
pd.DataFrame(cmp_rows).set_index(["split", "model"]).round(4)

auc     mrr  ndcg@5  ndcg@10  cold_recall@10  \
split model                                                                     
val   lgbm_full               0.7166  0.3630  0.3968   0.4533          0.6507   
      candidate_smoothed_ctr  0.6434  0.2975  0.3183   0.3773          0.2932   
      category_ctr            0.4903  0.2085  0.2154   0.2708          0.3324   
      content_smoothed_ctr    0.6449  0.2966  0.3175   0.3769          0.3701   
      nrms_g10                0.6365  0.3018  0.3254   0.3829          0.5803   
      popularity              0.4836  0.2192  0.2212   0.2755          0.0309   
      random                  0.5000  0.2182  0.2213   0.2779          0.2725   
test  lgbm_full               0.7141  0.3376  0.3695   0.4306          0.6054   
      candidate_smoothed_ctr  0.6468  0.2784  0.2937   0.3569          0.2489   
      category_ctr            0.4977  0.1945  0.1929   0.2549          0.2979   
      content_smoothed_ctr    0.6479  0.2775  0.2929   0.3563          0.2694   
      nrms_g10                0.6094  0.2658  0.2815   0.3439          0.5743   
      popularity              0.4608  0.1822  0.1792   0.2405          0.0287   
      random                  0.5000  0.2026  0.2034   0.2623          0.2282   

                              cold_hit@10  warm_recall@10  
split model                                                
val   lgbm_full                    0.6631          0.6870  
      candidate_smoothed_ctr       0.3017          0.6166  
      category_ctr                 0.3380          0.4984  
      content_smoothed_ctr         0.3795          0.6162  
      nrms_g10                     0.5949          0.6184  
      popularity                   0.0309          0.4897  
      random                       0.2814          0.4996  
test  lgbm_full                    0.6237          0.6727  
      candidate_smoothed_ctr       0.2593          0.5953  
      category_ctr                 0.3059          0.4808  
      content_smoothed_ctr         0.2784          0.5952  
      nrms_g10                     0.5926          0.5788  
      popularity                   0.0287          0.4666  
      random                       0.2378          0.4844

**Takeaway: headline comparison.** LightGBM is the best model on every metric on test. Its nDCG@10 is 0.431, against 0.357 for the best baseline and 0.344 for Group 10's model, and its AUC is 0.714 against 0.648 and 0.609. Its Cold Recall@10 (0.605) beats both the best baseline (0.298) and Group 10's model (0.574), which was until now the strongest model on cold articles. Combining behaviour, content and user matching does better than any one of them alone. These are point estimates; step 6 adds intervals.

## 6. Which feature groups the model leans on

Share of total split gain by group and the top individual features. Gain shows what the trees used, not what each group adds on its own; the ablation study in step 3d measures that.

In [6]:
imp = L.importance_by_group(full, GROUPS)
from IPython.display import display
display(imp.groupby("group")["gain_share"].sum().sort_values(ascending=False).round(4).to_frame())
imp.head(20).round(4)

,gain_share
group,
U,0.3376
T1,0.3278
T3,0.3200
T2,0.0147


,gain_share,group
prior_ctr,0.2443,T1
subcategory_code,0.1844,T3
text_similarity,0.1565,U
subcategory_affinity,0.0530,U
prior_exposures,0.0321,T1
category_affinity,0.0233,U
recent_exposures_6h,0.0194,T1
title_pc_04,0.0120,T3
title_pc_02,0.0117,T3
history_length,0.0111,U


**Takeaway: where the trees found signal.** Gain is split almost evenly between user matching U (34%), the article's own behaviour T1 (33%) and pure content T3 (32%); transferred behaviour T2 gets 1.5%. The top three features are prior CTR, subcategory, and the text similarity between the candidate and the user's history. Gain shows what the trees used, not what each group is worth. T2's topic-level click rates are largely recoverable from the subcategory label in T3 (the label alone explains 56–62% of their variance on validation rows, while no T2 column correlates above 0.32 with any U column), so the trees rarely need them. Step 3d tests that directly.

## 7. Binary-objective comparison

Same columns and tree shape, but binary log-loss with early stopping on validation AUC, so its scores are click probabilities. The paired bootstrap (1,000 resamples over impressions) gives the main model's gain over it with a 95% interval.

In [7]:
prob = L.fit(X_tr, m_tr["label"], X_va, m_va["label"], {**selected_params, "objective": "binary"}, seed=SEED)
bin_rows, gain_rows = {}, []
for split, (m, x) in data.items():
    sc = L.predict(prob, L.design(x, FULL_COLS))
    write_predictions(m["row_id"], sc, root=PRED_ROOT, data_kind=DATA_KIND, model_id="lgbm_full_binary",
                      config_id=SELECTED, split=split, schema_version=S.SCHEMA_VERSION,
                      extra={"objective": "binary", "best_iteration": prob.best_iteration, "selected_on": "val"})
    bin_rows[split] = E.evaluate(m, sc)
    if split != "train":
        main_pi, bin_pi = E.per_impression_metrics(m, main_scores[split]), E.per_impression_metrics(m, sc)
        for metric in ("ndcg@10", "cold_recall@10"):
            bs = E.paired_bootstrap(bin_pi[metric], main_pi[metric])
            gain_rows.append({"split": split, "metric": metric, "lambdarank - binary": bs["mean_diff"],
                              "95% CI": f"[{bs['ci_low']:+.4f}, {bs['ci_high']:+.4f}]"})
from IPython.display import display
display(pd.concat({"lambdarank (main)": pd.DataFrame(final).T[SHOW], "binary": pd.DataFrame(bin_rows).T[SHOW]}).round(4))
pd.DataFrame(gain_rows).set_index(["split", "metric"]).round(4)

auc     mrr  ndcg@5  ndcg@10  cold_recall@10  \
lambdarank (main) train  0.7768  0.4503  0.4957   0.5456          0.6943   
                  val    0.7166  0.3630  0.3968   0.4533          0.6507   
                  test   0.7141  0.3376  0.3695   0.4306          0.6054   
binary            train  0.7629  0.4207  0.4646   0.5182          0.5761   
                  val    0.7136  0.3558  0.3889   0.4468          0.5570   
                  test   0.7102  0.3341  0.3631   0.4257          0.4932   

                         cold_hit@10  warm_recall@10  
lambdarank (main) train       0.7111          0.7636  
                  val         0.6631          0.6870  
                  test        0.6237          0.6727  
binary            train       0.5948          0.7507  
                  val         0.5693          0.6847  
                  test        0.5114          0.6661

lambdarank - binary              95% CI
split metric                                                 
val   ndcg@10                      0.0065  [+0.0048, +0.0082]
      cold_recall@10               0.0936  [+0.0707, +0.1151]
test  ndcg@10                      0.0050  [+0.0031, +0.0069]
      cold_recall@10               0.1122  [+0.0868, +0.1384]

**Takeaway: objective.** Ranking within impressions beats predicting click probabilities on both splits, with intervals above zero: nDCG@10 +0.005 to +0.007 and Cold Recall@10 +0.09 to +0.11. The cold gain is the large one. A plausible reason is that the binary model learns that low-exposure articles are clicked less often overall, while LambdaRank only has to order candidates within one slate, where a cold article that fits the user can still rank high. This result is why LambdaRank is the main model.

## 8. What step 3c shows

* **LightGBM is the strongest non-deep model.** It beats every simple baseline and Group 10's network on all test metrics (section 5). The main deep model (step 4) is about level with it overall (test nDCG@10 0.428 vs 0.431) and higher on cold articles (Cold Recall@10 0.627 vs 0.605).
* **The ranking objective matters, mostly for cold articles.** LambdaRank adds about 0.1 Cold Recall@10 over the binary model and about 0.005 nDCG@10, with intervals above zero (section 7).
* **Signal comes from three places in roughly equal shares:** the article's own behaviour, the user–article match and the article's content. Transferred behaviour is barely used (section 6).
* Use `lgbm_full_binary` wherever calibrated probabilities are needed; LambdaRank scores only order candidates within an impression.
* Step 3d measures what each feature group is worth by retraining without it.

*Based on the run with feature schema v0.1 and the interim evaluator. The final numbers and intervals come from step 6.*